<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/credit-underwriting/lessons/P05-L02-reject-inference/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/credit-underwriting/lessons/P05-L02-reject-inference/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/credit-underwriting/lessons/P05-L02-reject-inference/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/credit-underwriting/lessons/P05-L02-reject-inference/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P05-L02 · Reject inference: what a scorecard learns when only approvals are observed

**You will build:** the machinery for a question every lender's data answers badly: *how good
is a scorecard on the applicants we declined?* You will impose an approve/decline policy on a
synthetic through-the-door population in which every applicant has an outcome, train a
scorecard on the approvals only, and measure how far its rank ordering (Gini) and its
calibration (actual over expected) fall on the whole population. Then you will build the three
standard answers to the problem, **augmentation** by weighting, **parcelling**, and **bounds**,
and score each against the truth that only a simulation can show you.

**Time:** ~2 hours · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download
· **Prerequisites:** `T00-L01-the-8gb-track` (what a declared compute budget means),
`P04-L01-validation-suite` (a reliability table and what actual over expected measures) and
`P05-L01-pd-vintages-and-macro` (outcomes that are missing because the data stop). Python and
numpy indexing; no pandas and no scikit-learn, every estimator here is yours or given.

By the end you will be able to:

1. Implement an approval policy with a fixed cut-off, a tie-aware Gini and a calibration table
   by score band, and use them to judge a scorecard on approvals against one trained on all.
2. Measure how far an approvals-only scorecard's Gini and actual-over-expected ratio fall on the
   whole through-the-door population as the policy gets stricter and as it leans on a variable
   the scorecard cannot see.
3. Implement augmentation weights (inverse probability of approval, capped) and the effective
   sample size they leave behind.
4. Implement parcelling (inferred outcomes by score band, with an uplift) and the weighted
   training set built from it, and measure how much of the gap each remedy recovers.
5. Implement bounds on the through-the-door default rate that state their assumption, and
   explain why no remedy can recover what the data never contained.

**The data are synthetic and the generator is in this notebook.** Nothing is downloaded. A
real lender's data hold the outcomes of the accepted loans only, which is exactly the
difficulty; a planted population lets you see the outcome of the applicant who was declined.
What you conclude here about *method* transfers; what you conclude about *these numbers*
describes no real lender.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`, `P04-L01`, `P05-L01`

**This lesson assumes you can already:**

- write numpy code that sorts an array, builds a boolean mask from the sorted order and uses it to index another array
- explain what a logistic regression predicts, and what a weighted fit means (a weight of 2 counts a row twice)
- read a Gini coefficient as a measure of how well scores rank outcomes, and an actual-over-expected ratio as a measure of whether the predicted rates are the right size
- say what an applicant's probability of default is, and compare it with the observed 0/1 outcome

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Indexing on ndarrays (NumPy user guide)](https://numpy.org/doc/stable/user/basics.indexing.html) · NumPy developers · *free* — the sections on boolean array indexing and index arrays: how a mask built in sorted order is applied back to the applicants in their original order
- [Mann-Whitney U test (Wikipedia)](https://en.wikipedia.org/wiki/Mann%E2%80%93Whitney_U_test) · Wikipedia contributors · *free* — the rank-sum calculation behind exercise 2, ties counted as one half, and its link to the area under the ROC curve and to the Gini coefficient
- [Best Practices for Responsible Machine Learning in Credit Scoring](https://arxiv.org/abs/2409.20536) · Valdrighi et al., arXiv 2409.20536 (2024) · *free* — the 'Reject Inference' section and its methods: sample bias, augmentation, extrapolation and fuzzy parcelling, the vocabulary of this lesson's section 8
- [Causal Inference: What If (chapter 12, IP weighting)](https://miguelhernan.org/whatifbook) · Miguel Hernan and James Robins · *free* — inverse-probability weighting explained from first principles: why weighting by 1 over the probability of being observed makes the observed group stand for the population; skim the causal notation
- [Intelligent Credit Scoring, 2nd edition (chapter 10, section 'Reject Inference')](https://www.wiley.com/en-us/intelligent-credit-scoring-building-and-implementing-better-credit-risk-scorecards-2nd-edition-p-9781119279150) · Naeem Siddiqi, Wiley (2017) · *paid* — the practitioner's account of reject inference (pages 215-236 in the publisher's table of contents): why it is done, simple and fuzzy augmentation and parcelling, the methods of sections 9 to 12

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import os
import signal
import sys
import threading
import time
import traceback
from typing import Any, Callable, NamedTuple

import numpy as np

import matplotlib
_INTERACTIVE = "ipykernel" in sys.modules
if not _INTERACTIVE:
    # Headless: a script run (including this repository's execution gate) must never try to
    # open a window. In Jupyter the default inline backend is already the right one.
    matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402  (backend must be chosen before this import)

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__,
      "· matplotlib", matplotlib.__version__)

# The population. Every applicant has five observed features, one hidden variable and an outcome
# (1 = defaults, 0 = repays). The lender sees the features of everyone who applies, and the
# outcome of the applicants it approved. SEED is the date this lesson was written.
SEED = 20261002
N_BOOK = 20_000          # applicants in the historical book the scorecard is trained on
N_TEST = 30_000          # a fresh through-the-door sample, every outcome known, for judging
TEST_OFFSET = 500_000    # the test sample's seed is the book's seed plus this
FEATURES = ("utilisation", "recent delinquencies", "debt-to-income", "short credit history",
            "recent enquiries")
BETA_TRUE = np.array([0.55, 0.45, 0.35, 0.30, 0.25])   # log-odds per standard deviation
THETA = 0.7              # log-odds per standard deviation of the hidden variable
KAPPA = 0.25             # extra risk at the extremes of utilisation (a curve a straight line misses)
B0 = -2.4                # intercept of the true log-odds

# The scenario the lesson keeps coming back to, and the grid it sweeps.
APPROVE_RATE = 0.50      # the old policy approved this share of applicants
HIDDEN_SHARE = 0.30      # share of the policy score that came from the hidden variable
NOISE_SHARE = 0.10       # share that was discretion unrelated to risk (overrides, branch habits)
RATES = (0.9, 0.7, 0.5, 0.3)
SHARES = (0.0, 0.2, 0.4, 0.6)
REPLICATES = 8           # books per cell of a sweep: one book is one draw
BANDS = 10               # score bands for parcelling and bounds
CAP = 20.0               # the most any approved applicant's augmentation weight may be


def _book_seed(k: int) -> int:
    """The seed of replicate book k. Fixed, so every run draws the same books."""
    return SEED + 7919 * k


def _show(fig: "matplotlib.figure.Figure") -> None:
    """Display a figure in Jupyter, or close it cleanly in a headless script run."""
    if _INTERACTIVE:
        plt.show()
    else:
        plt.close(fig)


_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("impose_policy",),
    "exercise 2": ("auc_gini",),
    "exercise 3": ("calibration_bands",),
    "exercise 4": ("augmentation_weights",),
    "exercise 5": ("effective_sample_size",),
    "exercise 6": ("parcel_bad_rates",),
    "exercise 7": ("stack_parcelled",),
    "exercise 8": ("banded_bounds",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run
S: dict[str, Any] = {}         # results the demos hand to one another
# Heavy demos (sweeps over many books) run when this file is the program, in a script or in a
# notebook kernel, and are skipped when a grader merely imports it to test your functions.
_RUNNING = __name__ == "__main__"


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (calibration_bands)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# How long an exercise's check may run before it counts as stuck, and how long any other cell that runs your
# code (a demonstration, a table) may: a right answer needs a small fraction of either, measured on the worked
# solution (internal/retrofit_guard.py, 10x or more). tools/verify_all.py reads both numbers.
_CHECK_SECONDS = 2
_DEMO_SECONDS = 45


class _OutOfTime(BaseException):
    """Raised inside a guarded block when its time is up. A BaseException, so an `except
    Exception:` in the code under test cannot swallow it, and it repeats every 0.25 s in case a
    bare `except:` does. The guard turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: int, memory_mib: float = 2048):
    """Turn a check that runs far too long into a failure, instead of a notebook that hangs.

    Uses the operating system's alarm, so it works where that exists (Linux, macOS, Colab) and is
    skipped, harmlessly, where it does not. An alarm that was already running (the grader's own
    time limit) keeps its deadline if it is due first and is put back, with the time it had
    left, when the check ends.
    """
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"still running after {seconds} s. Look for a loop whose stopping condition can never "
               "be met, or for a loop over every pair of applicants where a sort would do")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = (), limit: int = 0,
         heavy: bool = False) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero. A `heavy` demo is skipped when this file is imported rather than run.
    """
    if heavy and not _RUNNING:
        return
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        with _time_limit(limit or (_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS)):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def sigmoid(x: Any) -> Any:
    """Numerically safe logistic function 1 / (1 + exp(-x)). Given to you."""
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x)
    pos = x >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-x[pos]))
    ex = np.exp(x[~pos])
    out[~pos] = ex / (1.0 + ex)
    return out[()]

## 1. A population in which we can see who was declined

Read the generator: you are allowed to know exactly how this world works, which is the one
advantage a synthetic book has over real data. Each applicant has

- five **observed features** `z` (utilisation, delinquencies and so on, standardised, mildly
  correlated), which the lender records for everyone who applies;
- one **hidden variable** `h` that carries risk and that the old policy could see but the
  data warehouse did not keep: an underwriter's note, a manual review, a bureau attribute
  the new model will not be given;
- an **outcome** `y`, drawn from `logit(P(y=1)) = B0 + z . BETA_TRUE + THETA * h + KAPPA * (z0^2 - 1)`.

The old policy ranked applicants by a **policy score** (higher is riskier) built from three
parts: what the features say, the hidden variable, and noise (discretion unrelated to risk).
`hidden_share` and `noise_share` are the shares of the score's variance each part takes, and
the policy approves the least risky applicants until its approval rate is reached.
**That is the entire data-generating story for selection**, and it carries two warnings.

1. If the policy used only the features the scorecard sees (`hidden_share = 0`), the
   applicants it declined differ from the ones it approved *only through those features*.
   A statistician calls that selection on observables, and it is the benign case.
2. If the policy also used `h`, the declined applicants differ in something the data do not
   contain. That is selection on an unobservable, and nothing fitted to the approvals can
   see it. The notebook measures both, and does not tell you in advance which hurts more.

In [ ]:
class Applicants(NamedTuple):
    z: np.ndarray             # (n, 5) observed features, standardised
    hidden: np.ndarray        # (n,) the hidden variable: for checking only
    y: np.ndarray             # (n,) 1 = defaults, 0 = repays: known here for EVERY applicant
    policy_score: np.ndarray  # (n,) the old policy's risk score, higher is riskier


def simulate_applicants(n: int, seed: int, hidden_share: float,
                        noise_share: float = NOISE_SHARE) -> Applicants:
    """Deterministic through-the-door population. Same seed, same applicants, on any machine.

    The applicants and their outcomes depend on `seed` alone. `hidden_share` and `noise_share`
    change only the policy score, so two calls that differ in `hidden_share` describe the SAME
    people judged by two different old policies.
    """
    if hidden_share < 0 or noise_share < 0 or hidden_share + noise_share > 1:
        raise ValueError("hidden_share and noise_share must be non-negative and sum to at most 1")
    rng = np.random.default_rng(seed)
    common = rng.standard_normal((n, 1))
    z = np.sqrt(0.3) * common + np.sqrt(0.7) * rng.standard_normal((n, len(FEATURES)))
    hidden = rng.standard_normal(n)
    discretion = rng.standard_normal(n)
    draw = rng.random(n)
    eta = B0 + z @ BETA_TRUE + THETA * hidden + KAPPA * (z[:, 0] ** 2 - 1.0)
    y = (draw < sigmoid(eta)).astype(int)
    seen = z @ BETA_TRUE
    seen = seen / seen.std()
    policy = (np.sqrt(1.0 - hidden_share - noise_share) * seen
              + np.sqrt(hidden_share) * hidden + np.sqrt(noise_share) * discretion)
    return Applicants(z, hidden, y, policy)


BOOK = simulate_applicants(N_BOOK, SEED, HIDDEN_SHARE)
TEST = simulate_applicants(N_TEST, SEED + TEST_OFFSET, HIDDEN_SHARE)
print(f"the historical book: {N_BOOK:,} applicants; through the door, {BOOK.y.mean():.1%} of them "
      f"went on to default")
print(f"the test sample:     {N_TEST:,} fresh applicants, every outcome known, "
      f"{TEST.y.mean():.1%} default")
print(f"the old policy: approve {APPROVE_RATE:.0%}; its score is {HIDDEN_SHARE:.0%} hidden "
      f"variable, {NOISE_SHARE:.0%} discretion, {1 - HIDDEN_SHARE - NOISE_SHARE:.0%} features")
print("\nfeature            mean    std")
for _j, _name in enumerate(FEATURES):
    print(f"{_name:<22}{BOOK.z[:, _j].mean():>5.2f}  {BOOK.z[:, _j].std():>5.2f}")

## 2. Exercise 1 — `impose_policy()`

The first thing to build is the filter itself: given every applicant's policy score, decide
who was approved. The rule is a **fixed cut-off in approval rate**: approve the `k` least
risky applicants, where `k` is the approval rate times the number of applicants, **rounded to
the nearest whole applicant**. Two things go wrong here and both change who the scorecard
gets to learn from.

- A threshold written as `score <= cutoff` approves *every* applicant tied at the cut-off.
  With coarse scores that is a different policy from the one asked for. Break ties by
  position: among tied applicants the earlier ones are approved first.
- `int(rate * n)` truncates, and floating point does not multiply the way a calculator does:
  a rate of 0.29 on 100 applicants is not 29.0 exactly, so truncating can approve one
  applicant too few. Round to the nearest instead.

<details><summary>💡 Hint 1 — what to think about</summary>

Higher scores are riskier, so the approved applicants are the ones at the *low* end. If you
sorted the applicants by score, which positions in the sorted order would be approved? How
would you turn those positions back into a mask in the applicants' original order?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Work out how many applicants to approve, then take that many from the front of a *stable*
ordering of the scores (one that keeps earlier applicants first among equals) and mark them
in a boolean array of the original length. Reject, with `ValueError`, a score array that is
not a non-empty one-dimensional array of finite numbers, a rate outside `(0, 1]`, and a rate
so small that it approves nobody.
</details>

In [ ]:
def impose_policy(policy_score: np.ndarray, approve_rate: float) -> np.ndarray:
    """Approve the least risky applicants: a boolean mask, True where approved.

    Exactly `k = round(approve_rate * n)` applicants are approved, the `k` with the lowest
    `policy_score`. Applicants tied on score are taken in their original order, earliest first,
    so exactly `k` are approved even when scores tie.

    Raises ValueError if `policy_score` is not a non-empty 1-D array of finite numbers, if
    `approve_rate` is not in (0, 1], or if `k` would be zero.

    Example:
        >>> impose_policy(np.array([0.9, 0.1, 0.5, 0.3, 0.7]), 0.4).tolist()
        [False, True, False, True, False]
        >>> impose_policy(np.array([1.0, 1.0, 1.0, 1.0]), 0.5).tolist()
        [True, True, False, False]

    Returns:
        a boolean array of the same length as `policy_score`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_impose_policy() -> None:
    m = np.asarray(impose_policy(np.array([0.9, 0.1, 0.5, 0.3, 0.7]), 0.4))
    assert m.dtype == bool and m.tolist() == [False, True, False, True, False], (
        f"got {m.tolist()}, expected approvals at positions 1 and 3 (the two lowest scores). "
        "Approve the LOW scores: a higher policy score is a riskier applicant")
    tied = np.asarray(impose_policy(np.array([1.0, 1.0, 1.0, 1.0]), 0.5))
    assert tied.tolist() == [True, True, False, False], (
        f"four tied scores at rate 0.5 gave {tied.tolist()}; exactly two must be approved, the "
        "earlier two. A `score <= cutoff` test approves every tie at the cut-off")
    mixed = np.asarray(impose_policy(np.array([0.0, 1.0, 1.0, 1.0, 2.0]), 0.6))
    assert mixed.tolist() == [True, True, True, False, False], (
        f"got {mixed.tolist()}; three approvals means the lowest score and the first two of the "
        "three tied ones")
    n_ok = int(np.asarray(impose_policy(np.arange(100.0), 0.29)).sum())
    assert n_ok == 29, (
        f"rate 0.29 on 100 applicants approved {n_ok}, expected 29. In floating point 0.29 * 100 "
        "is a hair below 29, so int() truncates to 28: round to the nearest applicant")
    for bad_rate, bad_score in ((0.0, np.arange(5.0)), (1.5, np.arange(5.0)),
                                (0.1, np.arange(3.0)), (0.5, np.array([1.0, np.nan]))):
        try:
            impose_policy(bad_score, bad_rate)
        except ValueError:
            continue
        raise AssertionError(f"rate {bad_rate} on scores {bad_score.tolist()} must raise "
                             "ValueError (a rate outside (0, 1], one that approves nobody, or a "
                             "NaN score)")
    print("exercise 1 looks right — exactly k applicants approved, ties broken by position")

In [ ]:
_try("exercise 1", _check_impose_policy)

In [ ]:
def _show_lender_view() -> None:
    approved = impose_policy(BOOK.policy_score, APPROVE_RATE)
    S["approved"] = approved
    n_acc, n_rej = int(approved.sum()), int((~approved).sum())
    print(f"approved {n_acc:,} of {N_BOOK:,} ({n_acc / N_BOOK:.1%}); declined {n_rej:,}\n")
    print("                              what the lender can see      what happened")
    print(f"default rate, approvals       {BOOK.y[approved].mean():>20.1%}      "
          f"{BOOK.y[approved].mean():.1%}")
    print(f"default rate, declined        {'never observed':>20}      "
          f"{BOOK.y[~approved].mean():.1%}")
    print(f"default rate, everybody       {'never observed':>20}      {BOOK.y.mean():.1%}")
    print(f"\nmean of the hidden variable: approvals {BOOK.hidden[approved].mean():+.2f}, "
          f"declined {BOOK.hidden[~approved].mean():+.2f}, whole population "
          f"{BOOK.hidden.mean():+.2f}")
    print("\nthe lender's data hold the first column only. A scorecard trained on it never sees a")
    print("declined applicant's outcome, and the people it never sees are not a random sample:")
    print(f"the two groups differ on the hidden variable by "
          f"{BOOK.hidden[~approved].mean() - BOOK.hidden[approved].mean():.2f} standard deviations,")
    print("and by construction the scorecard cannot look at that variable.")


_try("what the lender sees", _show_lender_view, needs=("exercise 1",), heavy=True)

## 3. Fitting a scorecard, and two ways to judge it

The scorecard here is a logistic regression on the five features: the workhorse of consumer
credit, and a straight line in the features, so it cannot draw the curve `KAPPA` plants in the
truth. `fit_logistic` below is **given to you**: a weighted Newton fit with a tiny ridge, so
that exercises 4 to 7 can hand it weights. `ae_ratio` is the one-line calibration summary you
met in `P04-L01` and `P05-L01`: actual defaults over expected defaults, where 1.0 means the
portfolio's default count was predicted and above 1.0 means more applicants defaulted than the
scorecard said.

A scorecard is judged on two separate things, and the lesson's whole argument is that
selection damages them differently. **Discrimination** is whether risky applicants get higher
scores than safe ones, summarised by the **Gini coefficient**, `2 * AUC - 1`, where the AUC is
the chance that a randomly chosen defaulter outscores a randomly chosen repayer. **Calibration**
is whether the predicted default rates are the right size, which is what pricing and capital
are built from. You write both measures, starting with the Gini.

In [ ]:
def fit_logistic(X: np.ndarray, y: np.ndarray, weights: np.ndarray | None = None,
                 ridge: float = 1e-6) -> np.ndarray:
    """Weighted logistic regression by Newton's method. Given to you.

    Returns the coefficients, intercept first: `coef[0]` and then one per column of `X`. A
    weight of 0 removes a row, a weight of 2 counts it twice. Rows are applicants; `y` is 0/1.
    """
    X = np.asarray(X, dtype=float)
    A = np.column_stack([np.ones(len(X)), X])
    w = np.ones(len(X)) if weights is None else np.asarray(weights, dtype=float)
    y = np.asarray(y, dtype=float)
    coef = np.zeros(A.shape[1])
    for _ in range(50):
        p = sigmoid(A @ coef)
        grad = A.T @ (w * (p - y)) + ridge * coef
        hess = (A * (w * p * (1.0 - p))[:, None]).T @ A + ridge * np.eye(A.shape[1])
        step = np.linalg.solve(hess, grad)
        coef = coef - step
        if np.abs(step).max() < 1e-9:
            break
    return coef


def predict_pd(coef: np.ndarray, X: np.ndarray) -> np.ndarray:
    """Predicted probability of default for each row of `X`. Given to you."""
    return sigmoid(np.column_stack([np.ones(len(X)), np.asarray(X, dtype=float)]) @ coef)


def ae_ratio(p: np.ndarray, y: np.ndarray) -> float:
    """Actual over expected: observed defaults divided by the sum of predicted PDs. Given."""
    return float(np.sum(y) / np.sum(p))

## 4. Exercise 2 — `auc_gini()`

The AUC counts, over every (defaulter, repayer) pair, how often the defaulter has the higher
score, and a **tie counts one half**. Scores from a scorecard with a few bins tie constantly,
and a function that scores ties as zero, or as one, reports a discrimination the scorecard
does not have. The pair count is a rank statistic, so no loop over pairs is needed: with
`r_i` the rank of applicant `i` among all scores (tied scores sharing the *average* of the
ranks they occupy, ranks starting at 1), the number of won pairs is
`sum of r_i over defaulters - n1 * (n1 + 1) / 2` for `n1` defaulters, and the AUC is that over
`n1 * n0`. A loop over every pair of applicants is far too slow at this size, and the notebook
will stop it.

<details><summary>💡 Hint 1 — what to think about</summary>

Three scores `[1, 2, 2]` with outcomes `[0, 0, 1]`: the defaulter ties the second repayer and
beats the first. How many pairs are won, and how many are tied? What AUC does that give? Now
ask what rank each of the two tied scores should receive so that the formula above agrees.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Sort the scores, find the runs of equal values, and give every member of a run the average of
the positions the run covers. Put the ranks back in the original order, sum them over the
defaulters, subtract the smallest possible sum, and divide by the number of pairs. Raise
`ValueError` for unequal lengths, a `y` that is not all 0 or 1, or a `y` with only one class.
</details>

In [ ]:
def auc_gini(score: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    """Tie-aware AUC and the Gini coefficient `2 * AUC - 1`. Higher score = more likely y = 1.

    AUC is the probability that a randomly chosen y = 1 applicant has a higher score than a
    randomly chosen y = 0 applicant, counting a tie as one half.

    Raises ValueError if the arrays differ in length, are not 1-D, if `y` holds anything but
    0 and 1, or if `y` has only one class.

    Example (the defaulter beats one repayer and ties the other: AUC = (1 + 0.5) / 2):
        >>> auc_gini(np.array([1.0, 2.0, 2.0]), np.array([0, 0, 1]))
        (0.75, 0.5)

    Returns:
        (auc, gini) as Python floats.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_auc_gini() -> None:
    auc, gini = auc_gini(np.array([0.1, 0.2, 0.8, 0.9]), np.array([0, 0, 1, 1]))
    assert (auc, gini) == (1.0, 1.0), (
        f"a perfect ranking gave AUC {auc}, Gini {gini}; expected 1.0 and 1.0")
    auc, gini = auc_gini(np.array([0.9, 0.8, 0.2, 0.1]), np.array([0, 0, 1, 1]))
    assert (auc, gini) == (0.0, -1.0), (
        f"a perfectly reversed ranking gave AUC {auc}, Gini {gini}; expected 0.0 and -1.0 "
        "(higher score must mean more likely to default)")
    auc, gini = auc_gini(np.array([1.0, 2.0, 2.0]), np.array([0, 0, 1]))
    assert abs(auc - 0.75) < 1e-12 and abs(gini - 0.5) < 1e-12, (
        f"[1, 2, 2] with outcomes [0, 0, 1] gave AUC {auc}, expected 0.75: a tie counts one half, "
        "so the pair won plus half the pair tied, over two pairs")
    auc, _ = auc_gini(np.array([0.5, 0.5, 0.5, 0.5]), np.array([0, 1, 0, 1]))
    assert abs(auc - 0.5) < 1e-12, (
        f"four equal scores gave AUC {auc}, expected 0.5: a score that cannot tell the classes "
        "apart has no discrimination, and ties scored as wins or losses invent some")
    rng = np.random.default_rng(7)
    y_big = (rng.random(60_000) < 0.05).astype(int)
    auc, gini = auc_gini(rng.standard_normal(60_000) + y_big, y_big)
    assert 0.5 < auc < 1.0 and abs(gini - (2 * auc - 1)) < 1e-12, (
        f"60,000 applicants, defaulters shifted up by one standard deviation, gave AUC {auc} and "
        f"Gini {gini}: expected an AUC between 0.5 and 1 and Gini = 2 * AUC - 1")
    try:
        auc_gini(np.array([1.0, 2.0]), np.array([1, 1]))
    except ValueError:
        pass
    else:
        raise AssertionError("a y with only one class has no AUC: raise ValueError")
    print("exercise 2 looks right — ties count one half, no pair loop")

In [ ]:
_try("exercise 2", _check_auc_gini)

## 5. Exercise 3 — `calibration_bands()`

A portfolio-level A/E can hide a scorecard that is too high for the good applicants and too
low for the bad ones; a **calibration table** by score band shows where. Sort the applicants by
predicted PD, cut them into bands of (nearly) equal size, and in each band compare the mean
predicted PD with the default rate that actually happened. The cut is by *rank of the
prediction*, not by the outcome and not by position in the file, and when the applicants do not
divide evenly the first bands take one extra: the usual convention of `numpy.array_split`.
Ties in the prediction are ordered by position, as in exercise 1.

<details><summary>💡 Hint 1 — what to think about</summary>

Ten applicants into three bands: what sizes? Which end of the sorted order gets the extra
applicant? Then ask what changes if two applicants have exactly the same predicted PD and sit
on either side of a band boundary: does the outcome of one of them decide which band it lands
in?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Get a stable ordering of the predictions, split that ordering into `n_bands` consecutive
pieces with the near-equal rule, and for each piece take the mean prediction, the mean outcome
and the size. Raise `ValueError` for predictions outside `[0, 1]`, outcomes that are not 0 or
1, unequal lengths, or a number of bands below 1 or above the number of applicants.
</details>

In [ ]:
def calibration_bands(p: np.ndarray, y: np.ndarray, n_bands: int
                      ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Mean predicted PD, observed default rate and size of each band of equal-count bands.

    Applicants are ordered by `p` ascending (ties by original position), then cut into `n_bands`
    consecutive groups by `numpy.array_split`'s rule: the first `n % n_bands` groups hold one
    extra applicant.

    Raises ValueError if `p` leaves [0, 1], `y` is not 0/1, the lengths differ, or `n_bands` is
    below 1 or above the number of applicants.

    Example (five applicants, two bands of sizes 3 and 2):
        >>> mp, rate, size = calibration_bands(np.array([0.1, 0.9, 0.2, 0.8, 0.5]),
        ...                                    np.array([0, 1, 0, 0, 1]), 2)
        >>> [round(float(v), 4) for v in mp], rate.tolist(), size.tolist()
        ([0.2667, 0.85], [0.3333333333333333, 0.5], [3, 2])

    Returns:
        (mean_p, rate, size): three arrays of length `n_bands`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_calibration_bands() -> None:
    mp, rate, size = calibration_bands(np.array([0.1, 0.9, 0.2, 0.8, 0.5]),
                                       np.array([0, 1, 0, 0, 1]), 2)
    mp, rate, size = np.asarray(mp), np.asarray(rate), np.asarray(size)
    assert size.tolist() == [3, 2], (
        f"band sizes {size.tolist()}, expected [3, 2]: five applicants in two bands, the extra "
        "one goes to the FIRST band")
    assert np.allclose(mp, [0.8 / 3, 0.85]), (
        f"mean predicted PD per band {np.round(mp, 4).tolist()}, expected [0.2667, 0.85]. Bands "
        "are cut on the sorted PREDICTIONS, lowest first")
    assert np.allclose(rate, [1 / 3, 0.5]), (
        f"observed rate per band {np.round(rate, 4).tolist()}, expected [0.3333, 0.5]. The band "
        "of the three lowest predictions holds one defaulter; sorting by outcome or by position "
        "puts the wrong applicants together")
    p = np.array([0.3, 0.3, 0.3, 0.3])
    _, r2, _ = calibration_bands(p, np.array([1, 0, 1, 0]), 2)
    assert np.asarray(r2).tolist() == [0.5, 0.5], (
        f"four tied predictions gave band rates {np.asarray(r2).tolist()}; tied applicants keep "
        "their original order, so the bands are (1, 0) and (1, 0)")
    try:
        calibration_bands(np.array([0.2, 0.4]), np.array([0, 1]), 3)
    except ValueError:
        pass
    else:
        raise AssertionError("more bands than applicants must raise ValueError")
    print("exercise 3 looks right — equal-count bands on the sorted predictions")

In [ ]:
_try("exercise 3", _check_calibration_bands)

## 6. What the approvals-only scorecard gets wrong

Everything you have written so far is enough to measure the damage. The cell below trains
two scorecards on the same historical book, with the same five features: one on the
**approvals only**, which is all a real lender has, and one on **everybody**, which only a
simulation can build. It then judges both on the fresh through-the-door test sample, where
every outcome is known. The oracle is the yardstick: any gap between the two is what selection
cost, not what a straight line in five features cannot do.

Watch which row the lender could compute for itself. The first row is the approvals-only
scorecard judged on the approved applicants of the test sample, the only people whose
outcomes a real lender would observe. The rows below it are the truth.

In [ ]:
def judge(coef: np.ndarray, test: Applicants, approved_test: np.ndarray) -> dict[str, float]:
    """The scorecard `coef` on the test sample: Gini and A/E on everyone, on the approved
    applicants alone (the part a real lender can see) and on the declined ones alone."""
    p = predict_pd(coef, test.z)
    return {"gini": auc_gini(p, test.y)[1],
            "ae": ae_ratio(p, test.y),
            "gini_approved": auc_gini(p[approved_test], test.y[approved_test])[1],
            "ae_approved": ae_ratio(p[approved_test], test.y[approved_test]),
            "ae_declined": ae_ratio(p[~approved_test], test.y[~approved_test])}


def study_degradation(rate: float, hidden_share: float, seed: int, n_book: int = N_BOOK
                      ) -> dict[str, dict[str, float]]:
    """One historical book and one test sample: the approvals-only scorecard and the oracle."""
    book = simulate_applicants(n_book, seed, hidden_share)
    test = simulate_applicants(N_TEST, seed + TEST_OFFSET, hidden_share)
    approved = impose_policy(book.policy_score, rate)
    approved_test = impose_policy(test.policy_score, rate)
    return {"approved": judge(fit_logistic(book.z[approved], book.y[approved]), test, approved_test),
            "oracle": judge(fit_logistic(book.z, book.y), test, approved_test)}


def _show_damage() -> None:
    res = study_degradation(APPROVE_RATE, HIDDEN_SHARE, SEED)
    a, o = res["approved"], res["oracle"]
    S["damage"] = res
    print(f"{'':<54}{'Gini':>8}{'A/E':>8}")
    print(f"{'what the lender can measure:':<54}")
    print(f"{'  approvals-only scorecard, on the approved applicants':<54}"
          f"{a['gini_approved']:>8.3f}{a['ae_approved']:>8.2f}")
    print(f"{'the truth, on every applicant of the test sample:':<54}")
    print(f"{'  approvals-only scorecard':<54}{a['gini']:>8.3f}{a['ae']:>8.2f}")
    print(f"{'  scorecard trained on everyone (the oracle)':<54}{o['gini']:>8.3f}{o['ae']:>8.2f}")
    print(f"{'  approvals-only scorecard, declined applicants only':<54}"
          f"{'':>8}{a['ae_declined']:>8.2f}")
    print()
    if abs(a["ae_approved"] - 1) <= 0.10 < abs(a["ae"] - 1):
        print(f"on the approved applicants the scorecard's A/E is {a['ae_approved']:.2f}: it looks "
              f"calibrated, and across everybody it reads {a['ae']:.2f}. A scorecard is "
              "calibrated for the population it was fitted on, and the lender's own test "
              "sample is that population.")
    elif abs(a["ae_approved"] - 1) < abs(a["ae"] - 1):
        print(f"on the approved applicants the scorecard's A/E is {a['ae_approved']:.2f}, closer to 1 "
              f"than the {a['ae']:.2f} it reads across everybody: the lender's own test sample is "
              "the population the scorecard was fitted on.")
    if a["gini_approved"] < a["gini"]:
        print(f"the lender's own Gini ({a['gini_approved']:.3f}) is BELOW the Gini on everybody "
              f"({a['gini']:.3f}): the approvals are a narrow slice of the risk range, and a "
              "narrow slice is harder to rank. Selection shows up here as pessimism, not flattery.")
    print(f"the damage selection did to ranking is {o['gini'] - a['gini']:.3f} of Gini against the "
          f"oracle, and to calibration a factor of {a['ae'] / o['ae']:.2f} in A/E.")


_try("the damage", _show_damage, needs=("exercise 1", "exercise 2"), heavy=True)

Now the band-by-band view you wrote the function for. Each row is a tenth of the test sample,
ordered by the approvals-only scorecard's own prediction, so a well-calibrated scorecard has
its predicted and actual columns equal in every row.

In [ ]:
def _show_bands() -> None:
    book, test = BOOK, TEST
    approved = impose_policy(book.policy_score, APPROVE_RATE)
    p_acc = predict_pd(fit_logistic(book.z[approved], book.y[approved]), test.z)
    p_all = predict_pd(fit_logistic(book.z, book.y), test.z)
    mp_a, ra_a, _ = calibration_bands(p_acc, test.y, BANDS)
    mp_o, ra_o, size = calibration_bands(p_all, test.y, BANDS)
    print(f"{'band':>4}{'size':>7} | {'approvals-only':^23} | {'oracle':^23}")
    print(f"{'':>11} | {'predicted':>11}{'actual':>11} | {'predicted':>11}{'actual':>11}")
    for b in range(BANDS):
        print(f"{b + 1:>4}{int(size[b]):>7} | {mp_a[b]:>11.3f}{ra_a[b]:>11.3f} | "
              f"{mp_o[b]:>11.3f}{ra_o[b]:>11.3f}")
    gap_a = np.abs(ra_a - mp_a)
    gap_o = np.abs(ra_o - mp_o)
    worst = int(np.argmax(gap_a))
    print(f"\nthe approvals-only scorecard is furthest out in band {worst + 1} "
          f"(predicted {mp_a[worst]:.3f}, actual {ra_a[worst]:.3f}); the oracle's furthest band is "
          f"out by {gap_o.max():.3f}, the approvals-only one by {gap_a.max():.3f}.")
    fig, ax = plt.subplots(figsize=(5.2, 4.2))
    ax.plot([0, max(ra_a.max(), mp_a.max())], [0, max(ra_a.max(), mp_a.max())], color="grey",
            lw=0.8, label="perfect calibration")
    ax.plot(mp_a, ra_a, "o-", color="tab:red", label="approvals-only scorecard")
    ax.plot(mp_o, ra_o, "s-", color="tab:blue", label="trained on everyone")
    ax.set_xlabel("mean predicted PD in the band")
    ax.set_ylabel("actual default rate in the band")
    ax.legend(fontsize=8)
    fig.tight_layout()
    _show(fig)


_try("calibration by band", _show_bands, needs=("exercise 1", "exercise 3"), heavy=True)

## 7. How the damage grows

One book is one draw. The next cell sweeps the two dials the old policy had: **how strict it
was** (the approval rate) and **how much of its score came from the hidden variable**. For
every combination it simulates `REPLICATES` independent books, from fixed seeds, and reports
the mean and the lowest-to-highest range across them. The people are the same in every column
of a row, because only the policy score changes, so a difference between two columns is
the policy and not luck.

Two tables come out. The first is the **Gini lost on every applicant** against the scorecard
trained on everyone. The second is **A/E on every applicant**, with the range across books in
brackets. Read the shape of both before reading the printed conclusions beneath them.

In [ ]:
def _sweep_degradation() -> None:
    cell: dict[tuple[float, float], list[dict[str, dict[str, float]]]] = {}
    for rate in RATES:
        for share in SHARES:
            cell[(rate, share)] = [study_degradation(rate, share, _book_seed(k))
                                   for k in range(1, REPLICATES + 1)]
    S["sweep"] = cell

    def mean_of(rate: float, share: float, f: Callable[[dict], float]) -> float:
        return float(np.mean([f(r) for r in cell[(rate, share)]]))

    lost = {(r, s): mean_of(r, s, lambda d: d["oracle"]["gini"] - d["approved"]["gini"])
            for r in RATES for s in SHARES}
    ae = {(r, s): mean_of(r, s, lambda d: d["approved"]["ae"]) for r in RATES for s in SHARES}
    head = "".join(f"{f'hidden {s:.0%}':>18}" for s in SHARES)
    print("Gini lost on every applicant (oracle minus approvals-only), mean of "
          f"{REPLICATES} books\n" + f"{'approve rate':<14}" + head)
    for r in RATES:
        print(f"{r:<14.0%}" + "".join(f"{lost[(r, s)]:>18.3f}" for s in SHARES))
    print(f"\nA/E on every applicant, mean (lowest-highest across {REPLICATES} books)\n"
          f"{'approve rate':<14}" + head)
    for r in RATES:
        row = f"{r:<14.0%}"
        for s in SHARES:
            vals = [d["approved"]["ae"] for d in cell[(r, s)]]
            row += f"{np.mean(vals):.2f} ({min(vals):.2f}-{max(vals):.2f})".rjust(18)
        print(row)
    loose, strict = RATES[0], RATES[-1]
    grows = sum(lost[(strict, s)] > lost[(loose, s)] for s in SHARES)
    print(f"\nthe Gini lost is larger at the strictest policy than at the loosest in {grows} of "
          f"{len(SHARES)} columns.")
    benign, hidden_cols = SHARES[0], SHARES[1:]
    worse = sum(ae[(r, s)] > ae[(r, benign)] for r in RATES for s in hidden_cols)
    print(f"a policy that leaned on the hidden variable is worse calibrated than one that did "
          f"not, at the same approval rate, in {worse} of {len(RATES) * len(hidden_cols)} cells.")
    for r in RATES:
        best = max(SHARES, key=lambda s: ae[(r, s)])
        print(f"  at approve rate {r:.0%}: the worst A/E is {ae[(r, best)]:.2f}, at hidden share "
              f"{best:.0%}")
    print(f"\nthe largest Gini lost anywhere on the grid is {max(lost.values()):.3f}; the largest "
          f"A/E is {max(ae.values()):.2f} where the oracle's is {np.mean([d['oracle']['ae'] for d in cell[(loose, benign)]]):.2f}.")


_try("sweep of the damage", _sweep_degradation, needs=("exercise 1", "exercise 2"), limit=100,
     heavy=True)

One more dial: the size of the book. If the damage were noise, more approved loans would shrink
it. The cell below refits the approvals-only scorecard on books of each size in `BOOK_SIZES`,
`BOOKS_PER_SIZE` books of each from fixed seeds, at the scenario's approval rate, and judges each
on its test sample.

In [ ]:
BOOK_SIZES = (5_000, 20_000, 80_000)
BOOKS_PER_SIZE = 4


def _bigger_book() -> None:
    shares = (SHARES[0], HIDDEN_SHARE)
    print(f"A/E on every applicant at approve rate {APPROVE_RATE:.0%}, mean (lowest-highest) of "
          f"{BOOKS_PER_SIZE} books")
    print(f"{'book size':<12}" + "".join(f"{f'hidden {s:.0%}':>22}" for s in shares))
    res: dict[tuple[int, float], list[float]] = {}
    for n in BOOK_SIZES:
        row = f"{n:<12,}"
        for share in shares:
            res[(n, share)] = [study_degradation(APPROVE_RATE, share, _book_seed(k), n_book=n)
                               ["approved"]["ae"] for k in range(1, BOOKS_PER_SIZE + 1)]
            v = res[(n, share)]
            row += f"{np.mean(v):.2f} ({min(v):.2f}-{max(v):.2f})".rjust(22)
        print(row)
    small, big = BOOK_SIZES[0], BOOK_SIZES[-1]
    print()
    for share in shares:
        a, b = res[(small, share)], res[(big, share)]
        line = (f"hidden {share:.0%}: from {small:,} to {big:,} applicants the range across books went "
                f"from {max(a) - min(a):.2f} wide to {max(b) - min(b):.2f}, and the mean A/E from "
                f"{np.mean(a):.2f} to {np.mean(b):.2f}")
        if min(b) > 1.10:
            narrower = max(b) - min(b) < 0.75 * (max(a) - min(a))
            line += ((": more precise, and " if narrower else ": ")
                     + f"every book of {big:,} applicants is still far from calibrated. That is a bias")
        print(line + ".")


_try("a bigger book", _bigger_book, needs=("exercise 1", "exercise 2"), limit=60, heavy=True)

Two readings to test against the tables you just printed, because the notebook printed the
evidence for each. **Is the first column zero?** If the old policy used only the features the
scorecard sees, a gap that remains comes from extrapolation: a straight line fitted to the
safe end of a curve does not know how the curve bends at the risky end, and the gap should
grow with the policy's strictness. **Are the hidden columns worse, and does more data cure
them?** The approved applicants who look risky on the features are, on average, the ones the
old policy's hidden information said were safe, so the features' effect is understated and the
intercept sits too low. That is a bias and not a variance: a bigger book can make it more
precise, but not smaller. The printed lines above say which of these your run supports.

## 8. Reject inference: three answers to a missing column

**Reject inference** is the name for methods that try to put the declined applicants back into
the picture. As of 2 October 2026 a 2024 survey of the field calls the problem *sample bias*,
the case where the training data comprise only previously accepted clients. It defines
*augmentation* as adjusting the weights of accepted applicants to represent the rejected
population, notes that some authors use *parcelling* for the plain version of inferring rejected
applicants' outcomes from a model fitted on the accepted ones, and warns that the plain version
assumes the rejected would behave exactly like the accepted. Because the names vary, **this
lesson fixes its own**:

- **Augmentation** reweights the approvals. No declined applicant's outcome is invented.
- **Parcelling** gives each declined applicant an inferred outcome, drawn from the approvals'
  default rate in the same score band, scaled by an **uplift** that says how much riskier the
  declined are than approvals with the same score.
- **Bounds** make no attempt to find the answer. They say what range of answers each stated
  assumption allows.

The measuring rod is the oracle you already trained. The question for each remedy is how much
of the oracle's calibration and ranking it recovers, and the answer depends on the hidden
variable, which none of the three can see. First, what a remedy is actually given: the features
of the declined applicants, and nothing else about them.

In [ ]:
def _show_declined_features() -> None:
    approved = impose_policy(BOOK.policy_score, APPROVE_RATE)
    print(f"{'feature':<24}{'approved':>10}{'declined':>10}{'gap in sd':>11}")
    for j, name in enumerate(FEATURES):
        a, d = BOOK.z[approved, j].mean(), BOOK.z[~approved, j].mean()
        print(f"{name:<24}{a:>10.2f}{d:>10.2f}{(d - a) / BOOK.z[:, j].std():>11.2f}")
    print(f"{'(hidden variable)':<24}{BOOK.hidden[approved].mean():>10.2f}"
          f"{BOOK.hidden[~approved].mean():>10.2f}{'not given':>11}")
    print("\nthe declined applicants' features are on file, so a remedy can reweight by them. "
          "The last row is not on file for anyone, and no remedy below is given it.")


_try("what a remedy is given", _show_declined_features, needs=("exercise 1",), heavy=True)

## 9. Exercise 4 — `augmentation_weights()`

The lender has the *features* of every applicant it ever saw, declined ones included, so it
can fit a second model that predicts **whether an applicant was approved**. Call its output
`p_approve`. An approved applicant whose `p_approve` is 0.25 is one of a kind of applicant of
whom only a quarter were let in, so each such approval **stands for four** like itself. The
weight is the inverse of the probability of approval. Applicants who were declined carry no
outcome and get weight 0.

Inverse weights have a hazard, and the exercise grades it. An approval with a tiny
`p_approve` is an applicant the old policy almost never let in, and a weight of a thousand
lets one outcome decide the fit. **Cap** the weight: no approved applicant may count for more
than `cap` applicants. A cap is a bias-variance trade you choose, not a detail.

<details><summary>💡 Hint 1 — what to think about</summary>

What should an approved applicant with `p_approve` of exactly 1 weigh? And one whose
`p_approve` is 0.0, which the model says should never have been approved? What does `1 / 0.0`
do in numpy, and what would you want the weight to be instead?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Start from an array of zeros. For the approved applicants only, put in the smaller of the
inverse probability and the cap, taking care that a probability of zero cannot divide. Reject
a `p_approve` outside `[0, 1]`, arrays that differ in length, and a cap below 1, since no
approved applicant can stand for fewer than one applicant.
</details>

In [ ]:
def augmentation_weights(p_approve: np.ndarray, approved: np.ndarray, cap: float) -> np.ndarray:
    """Inverse-probability-of-approval weights, capped. Declined applicants get weight 0.

    For an approved applicant the weight is `1 / p_approve`, but never more than `cap`; a
    `p_approve` of 0 for an approved applicant gets the cap, without a division by zero.
    The weights are NOT rescaled: they add up, roughly, to the number of applicants the
    approvals stand for.

    Raises ValueError if the arrays are not 1-D and of equal length, if any `p_approve` is
    outside [0, 1] or not finite, or if `cap` is below 1.

    Example (the third applicant was declined; the second's weight of 4 is held to 3):
        >>> augmentation_weights(np.array([0.5, 0.25, 0.9]), np.array([True, True, False]), 3.0)
        array([2., 3., 0.])

    Returns:
        a float array, one weight per applicant.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_augmentation_weights() -> None:
    w = np.asarray(augmentation_weights(np.array([0.5, 0.25, 0.9]),
                                        np.array([True, True, False]), 3.0), dtype=float)
    assert w.shape == (3,) and np.allclose(w, [2.0, 3.0, 0.0]), (
        f"got {w.tolist()}, expected [2.0, 3.0, 0.0]: 1 / 0.5 = 2; 1 / 0.25 = 4 held to the cap "
        "of 3; the declined applicant has weight 0, whatever its p_approve")
    w = np.asarray(augmentation_weights(np.array([1.0, 0.0, 0.2]),
                                        np.array([True, True, True]), 10.0), dtype=float)
    assert np.allclose(w, [1.0, 10.0, 5.0]), (
        f"got {w.tolist()}, expected [1.0, 10.0, 5.0]: an approved applicant with p_approve 0 "
        "gets the cap, not an infinity or a NaN")
    assert np.isfinite(w).all(), "a weight came back infinite or NaN"
    w2 = np.asarray(augmentation_weights(np.array([0.5, 0.5]), np.array([True, False]), 5.0))
    assert w2.tolist() == [2.0, 0.0], (
        f"got {w2.tolist()}: the weights are not rescaled, and a declined applicant's is 0")
    for p_bad, cap in ((np.array([0.5, 1.2]), 5.0), (np.array([0.5, 0.5]), 0.5)):
        try:
            augmentation_weights(p_bad, np.array([True, True]), cap)
        except ValueError:
            continue
        raise AssertionError("a p_approve outside [0, 1], or a cap below 1, must raise ValueError")
    print("exercise 4 looks right — each approval stands for 1 / p_approve applicants, capped")

In [ ]:
_try("exercise 4", _check_augmentation_weights)

## 10. Exercise 5 — `effective_sample_size()`

Weights are not free. If a few approvals carry most of the weight, the weighted fit behaves as
though it had been given only a handful of loans. The standard summary is **Kish's effective
sample size**: `(sum of w) ** 2 / (sum of w ** 2)`. Equal weights give back the number of
loans; one weight a hundred times the others collapses it towards one. It is the price tag on
exercise 4, and it is what tells you whether the cap is too loose.

<details><summary>💡 Hint 1 — what to think about</summary>

Four weights of 1 should be worth four loans. One weight of 1 and three of 0 should be worth
one. What does the formula give for `[3, 1]`, and does it feel right that three loans' worth of
weight on one applicant buys less than two loans?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Two sums and a division. Zero weights add nothing to either sum, so declined applicants can
stay in the array. Reject, with `ValueError`, an array that is empty or not 1-D, any negative or
non-finite weight, and weights that all sum to zero.
</details>

In [ ]:
def effective_sample_size(w: np.ndarray) -> float:
    """Kish's effective sample size `(sum w)^2 / sum w^2` of a vector of weights.

    Raises ValueError if `w` is empty or not 1-D, has a negative or non-finite entry, or sums
    to zero. Zero weights are allowed.

    Example (a weight of 3 beside a weight of 1 is worth 1.6 loans, not 2):
        >>> effective_sample_size(np.array([3.0, 1.0]))
        1.6

    Returns:
        the effective sample size, a Python float between 1 and the number of nonzero weights.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_effective_sample_size() -> None:
    cases = ((np.array([1.0, 1.0, 1.0, 1.0]), 4.0), (np.array([1.0, 0.0, 0.0]), 1.0),
             (np.array([3.0, 1.0]), 1.6), (np.array([2.0, 2.0, 0.0, 2.0]), 3.0))
    for w, want in cases:
        got = effective_sample_size(w)
        assert abs(got - want) < 1e-9, (
            f"weights {w.tolist()} gave {got}, expected {want}. The formula is (sum of w) squared "
            "over the sum of w squared; dividing by the count, or taking the square root of one "
            "of the sums, gives other numbers")
    assert abs(effective_sample_size(np.array([5.0, 5.0])) - effective_sample_size(
        np.array([1.0, 1.0]))) < 1e-9, "rescaling every weight must not change the effective size"
    try:
        effective_sample_size(np.array([0.0, 0.0]))
    except ValueError:
        pass
    else:
        raise AssertionError("weights that sum to zero must raise ValueError")
    print("exercise 5 looks right — the effective size is (sum w)^2 / sum w^2")

In [ ]:
_try("exercise 5", _check_effective_sample_size)

Before the first remedy runs, two helpers the demos share are given to you. `fit_base` builds
one historical book, its test sample, the old policy's approvals, and the two scorecards:
approvals-only and the oracle. Rejected applicants' outcomes are hidden from everything a
remedy sees (`y_seen` is zero for them); only the final judging uses the truth.

In [ ]:
class Base(NamedTuple):
    book: Applicants
    test: Applicants
    approved: np.ndarray        # which book applicants the old policy approved
    approved_test: np.ndarray   # the same policy on the test sample
    y_seen: np.ndarray          # the book's outcomes with the declined applicants' set to 0
    coef_acc: np.ndarray        # scorecard trained on the approvals only
    coef_all: np.ndarray        # the oracle, trained on everyone


def fit_base(rate: float, hidden_share: float, seed: int) -> Base:
    """Simulate a book and a test sample, apply the old policy and fit the two scorecards."""
    book = simulate_applicants(N_BOOK, seed, hidden_share)
    test = simulate_applicants(N_TEST, seed + TEST_OFFSET, hidden_share)
    approved = impose_policy(book.policy_score, rate)
    approved_test = impose_policy(test.policy_score, rate)
    y_seen = np.where(approved, book.y, 0)
    return Base(book, test, approved, approved_test, y_seen,
                fit_logistic(book.z[approved], y_seen[approved]), fit_logistic(book.z, book.y))


def augmented_scorecard(base: Base) -> tuple[np.ndarray, np.ndarray]:
    """Augmentation: model P(approved | features) on EVERY applicant, weight the approvals by
    its inverse (capped), refit. Returns the coefficients and the weights."""
    coef_approve = fit_logistic(base.book.z, base.approved.astype(float))
    w = augmentation_weights(predict_pd(coef_approve, base.book.z), base.approved, CAP)
    return fit_logistic(base.book.z, base.y_seen, w), w


def _show_augmentation() -> None:
    print(f"{'':<34}{'approvals-only':>16}{'augmented':>12}{'oracle':>10}")
    for share in (HIDDEN_SHARE, 0.0):
        base = fit_base(APPROVE_RATE, share, SEED)
        coef_aug, w = augmented_scorecard(base)
        res = {"approvals-only": judge(base.coef_acc, base.test, base.approved_test),
               "augmented": judge(coef_aug, base.test, base.approved_test),
               "oracle": judge(base.coef_all, base.test, base.approved_test)}
        label = f"hidden share {share:.0%}"
        print(f"{label + ', Gini on everyone':<34}" + "".join(
            f"{res[k]['gini']:>{wd}.3f}" for k, wd in (("approvals-only", 16), ("augmented", 12),
                                                       ("oracle", 10))))
        print(f"{'  A/E on everyone':<34}" + "".join(
            f"{res[k]['ae']:>{wd}.2f}" for k, wd in (("approvals-only", 16), ("augmented", 12),
                                                    ("oracle", 10))))
        print(f"{'  A/E on the declined':<34}" + "".join(
            f"{res[k]['ae_declined']:>{wd}.2f}" for k, wd in (("approvals-only", 16),
                                                              ("augmented", 12), ("oracle", 10))))
        S.setdefault("aug", {})[share] = (res, w)
    res, w = S["aug"][HIDDEN_SHARE]
    n_acc = int((w > 0).sum())
    print(f"\n{n_acc:,} approvals stand for {w.sum():,.0f} applicants (the book had {N_BOOK:,}); "
          f"largest weight {w.max():.1f}, {int((w >= CAP).sum()):,} approvals held at the cap of "
          f"{CAP:.0f}.")
    print(f"effective sample size {effective_sample_size(w):,.0f} against {n_acc:,} approvals: "
          f"the weights cost {1 - effective_sample_size(w) / n_acc:.0%} of the sample.")
    for share, label in ((0.0, "the old policy used only the features"),
                         (HIDDEN_SHARE, f"{HIDDEN_SHARE:.0%} of its score was the hidden variable")):
        r = S["aug"][share][0]
        acc, aug = r["approvals-only"]["ae"], r["augmented"]["ae"]
        print(f"when {label}, augmentation moves A/E on everyone from {acc:.2f} to {aug:.2f}, "
              f"{1 - np.log(aug) / np.log(acc):.0%} of the gap on a log scale.")
    print(f"one book is one draw: section 14 repeats this over {REPLICATES} books.")


_try("augmentation", _show_augmentation,
     needs=("exercise 1", "exercise 2", "exercise 4", "exercise 5"), heavy=True)

Augmentation can only correct for what its approval model can see. It models approval from the
features, so it reweights the approvals towards the declined applicants *who looked like them
on the features*. If the old policy declined people for a reason that is not in the features,
there is nothing for the weights to attach to. The weights also need **overlap**: if the old
policy approved *only* applicants below a hard line on the features, nobody on the other side
was ever approved, `p_approve` is 0 or 1, and the weights are all 1. Run the check below to
see how much of that overlap this old policy left.

In [ ]:
def _show_overlap() -> None:
    base_noisy = simulate_applicants(N_BOOK, SEED, HIDDEN_SHARE, noise_share=NOISE_SHARE)
    base_hard = simulate_applicants(N_BOOK, SEED, 0.0, noise_share=0.0)
    print(f"{'old policy':<42}{'approvals with p_approve < 0.95':>34}")
    for name, pop in ((f"features, hidden variable and {NOISE_SHARE:.0%} discretion", base_noisy),
                      ("features only, no discretion (a hard line)", base_hard)):
        ok = impose_policy(pop.policy_score, APPROVE_RATE)
        coef = fit_logistic(pop.z, ok.astype(float))
        p = predict_pd(coef, pop.z)
        grey = float(np.mean(p[ok] < 0.95))
        print(f"{name:<42}{grey:>34.1%}")
    print("\nthe second row is the lender whose approval rule is a hard threshold on a score it "
          "records: the share above is how much of the approved population has any chance of "
          "standing for an applicant who was declined.")


_try("overlap", _show_overlap, needs=("exercise 1",), heavy=True)

## 11. Exercise 6 — `parcel_bad_rates()`

Parcelling infers an outcome for each declined applicant. The recipe has four steps and the
first three are mechanical.

1. Score everyone with the **approvals-only scorecard**.
2. Cut the approvals into `n_bands` score bands of equal size and cut the declined applicants
   by the **same edges**, taken from the approvals. `score_bands` is given and does exactly
   this: a score equal to an edge belongs to the upper band, and a declined applicant riskier
   than every approval lands in the top band, where the approvals' rate is the best the data
   can offer.
3. In each band, the approvals' default rate is the baseline for the declined applicants there.
4. Multiply by the **uplift**, an assumption that the declined are riskier than the approvals
   they sit beside, and cap the result at 1.

Step 4 carries the whole method. An uplift of 1 says the declined behave exactly like approvals
with the same score, which the survey cited above warns against, and no data can estimate any
other value: the real default rate of the declined is the thing that is missing. In practice
it is a judgement. Here the simulation knows the answer, and the notebook will show it to you.

<details><summary>💡 Hint 1 — what to think about</summary>

Cut the *approvals* into bands, not the declined applicants. What would the bands mean if the
declined were cut into their own tenths, and what rate would you then have to compare with?
What happens to a declined applicant whose score is below the lowest approval?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Use `score_bands` to put the approvals and then the declined into bands, take the default rate
of the approvals in each band, look up each declined applicant's band, multiply by the uplift
and cap at 1. Reject, with `ValueError`, an uplift below 1, fewer than one band, a `y_acc`
that is not 0/1, arrays of unequal length, and a band that holds no approval at all.
</details>

In [ ]:
def score_bands(score_ref: np.ndarray, score: np.ndarray, n_bands: int) -> np.ndarray:
    """Band index 0..n_bands-1 of each `score`, with edges at the equal-count quantiles of
    `score_ref`. A score equal to an edge is in the upper band; scores beyond the reference's
    range fall in the first or the last band. Given to you."""
    edges = np.quantile(np.asarray(score_ref, dtype=float), np.arange(1, n_bands) / n_bands)
    return np.searchsorted(edges, np.asarray(score, dtype=float), side="right")


def parcel_bad_rates(score_acc: np.ndarray, y_acc: np.ndarray, score_rej: np.ndarray,
                     n_bands: int, uplift: float) -> np.ndarray:
    """Inferred probability of default for each declined applicant, by score band.

    Bands come from the APPROVALS' scores (`score_bands(score_acc, ., n_bands)`). A declined
    applicant's probability is `min(1, uplift * (the approvals' default rate in its band))`.

    Raises ValueError if the arrays have different lengths or are not 1-D, if `y_acc` is not 0/1,
    if `n_bands` is below 1, if `uplift` is below 1, or if some band holds no approval.

    Example (edges at 2.5; the score 2.5 sits in the upper band, whose approvals defaulted at
    0.5, and 0.5 * 1.5 = 0.75):
        >>> parcel_bad_rates(np.array([1.0, 2.0, 3.0, 4.0]), np.array([0, 0, 1, 0]),
        ...                  np.array([0.5, 2.5, 9.0]), 2, 1.5)
        array([0.  , 0.75, 0.75])

    Returns:
        a float array, one probability per declined applicant.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_parcel_bad_rates() -> None:
    sa = np.array([1.0, 2.0, 3.0, 4.0])
    ya = np.array([0, 0, 1, 0])
    got = np.asarray(parcel_bad_rates(sa, ya, np.array([0.5, 2.5, 9.0]), 2, 1.5), dtype=float)
    assert got.shape == (3,) and np.allclose(got, [0.0, 0.75, 0.75]), (
        f"got {np.round(got, 4).tolist()}, expected [0.0, 0.75, 0.75]. The edge between the two "
        "bands is the median of the APPROVALS' scores, 2.5; a score on the edge belongs to the "
        "upper band; each rate is the approvals' band rate times the uplift")
    got = np.asarray(parcel_bad_rates(sa, ya, np.array([3.5, 4.5]), 2, 4.0), dtype=float)
    assert np.allclose(got, [1.0, 1.0]), (
        f"got {np.round(got, 4).tolist()}: 4.0 * 0.5 is 2.0, and a probability cannot exceed 1. "
        "Cap the product at 1")
    got = np.asarray(parcel_bad_rates(sa, ya, np.array([100.0, -100.0]), 2, 1.0), dtype=float)
    assert np.allclose(got, [0.5, 0.0]), (
        f"got {np.round(got, 4).tolist()}: a declined applicant riskier than every approval takes "
        "the top band's rate, and one safer than every approval takes the bottom band's")
    try:
        parcel_bad_rates(sa, ya, np.array([1.0]), 2, 0.5)
    except ValueError:
        pass
    else:
        raise AssertionError("an uplift below 1 must raise ValueError: it would say the declined "
                             "are safer than the approvals")
    print("exercise 6 looks right — the approvals' band rate, scaled by the uplift, capped at 1")

In [ ]:
_try("exercise 6", _check_parcel_bad_rates)

## 12. Exercise 7 — `stack_parcelled()`

An inferred probability of 0.3 is not a label. The fuzzy way to use it is to **duplicate** each
declined applicant: once as a defaulter with weight 0.3 and once as a repayer with weight 0.7.
The weighted logistic fit then sees 0.3 of a defaulter and 0.7 of a repayer, whose combined
weight is exactly one applicant. The approvals keep their observed outcomes with weight 1. The
function returns the stacked training set that `fit_logistic` takes as `X`, `y` and `weights`,
and the notebook trains the parcelled scorecard on it.

<details><summary>💡 Hint 1 — what to think about</summary>

If every inferred probability were 0 or 1, what should the stacked weights for one declined
applicant look like? Whatever you decide, the two weights for one declined applicant must add
up to what?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Stack three blocks: the approvals, the declined with outcome 1 and weight `p_bad`, and the
declined again with outcome 0 and weight `1 - p_bad`. Keep the order the docstring says. Reject
a `p_bad` outside `[0, 1]` or of the wrong length, and a feature matrix whose width differs
between approvals and declined.
</details>

In [ ]:
def stack_parcelled(X_acc: np.ndarray, y_acc: np.ndarray, X_rej: np.ndarray, p_bad: np.ndarray
                    ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """The weighted training set for a parcelled scorecard.

    Rows, in this order: every approval (its observed `y_acc`, weight 1); every declined
    applicant as a defaulter (y = 1, weight `p_bad`); every declined applicant as a repayer
    (y = 0, weight `1 - p_bad`). So the result has `n_acc + 2 * n_rej` rows.

    Raises ValueError if `p_bad` has a different length from `X_rej`, leaves [0, 1], or if the
    two feature matrices are not 2-D with the same number of columns, or `y_acc` and `X_acc`
    differ in length.

    Example (one approval, two declined applicants with inferred probabilities 0.25 and 1):
        >>> X, y, w = stack_parcelled(np.array([[1.0]]), np.array([1]),
        ...                           np.array([[2.0], [3.0]]), np.array([0.25, 1.0]))
        >>> X.ravel().tolist(), y.tolist(), w.tolist()
        ([1.0, 2.0, 3.0, 2.0, 3.0], [1.0, 1.0, 1.0, 0.0, 0.0], [1.0, 0.25, 1.0, 0.75, 0.0])

    Returns:
        (X, y, weights): a 2-D float array, and two 1-D float arrays of its length.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_stack_parcelled() -> None:
    X, y, w = stack_parcelled(np.array([[1.0]]), np.array([1]), np.array([[2.0], [3.0]]),
                              np.array([0.25, 1.0]))
    X, y, w = np.asarray(X), np.asarray(y), np.asarray(w)
    assert X.shape == (5, 1) and y.shape == (5,) and w.shape == (5,), (
        f"shapes {X.shape}, {y.shape}, {w.shape}: one approval and two declined applicants make "
        "1 + 2 * 2 = 5 rows")
    assert X.ravel().tolist() == [1.0, 2.0, 3.0, 2.0, 3.0] and y.tolist() == [1, 1, 1, 0, 0], (
        f"rows {X.ravel().tolist()}, outcomes {y.tolist()}; expected the approval, then the "
        "declined as defaulters (y = 1), then the declined again as repayers (y = 0)")
    assert np.allclose(w, [1.0, 0.25, 1.0, 0.75, 0.0]), (
        f"weights {w.tolist()}, expected [1.0, 0.25, 1.0, 0.75, 0.0]: the defaulter copy carries "
        "p_bad and the repayer copy 1 - p_bad, so each declined applicant weighs exactly 1 in "
        "total")
    try:
        stack_parcelled(np.array([[1.0]]), np.array([1]), np.array([[2.0]]), np.array([1.5]))
    except ValueError:
        pass
    else:
        raise AssertionError("a p_bad above 1 must raise ValueError")
    print("exercise 7 looks right — each declined applicant counts once, split by its p_bad")

In [ ]:
_try("exercise 7", _check_stack_parcelled)

In [ ]:
def planted_uplift(base: Base, n_bands: int = BANDS) -> float:
    """How much riskier the declined REALLY are than approvals in the same score band: their
    actual defaults divided by what the approvals' band rates would predict. It reads the
    declined applicants' true outcomes, so only a simulation can give it. Given to you."""
    s = predict_pd(base.coef_acc, base.book.z)
    s_acc = s[base.approved]
    band_acc = score_bands(s_acc, s_acc, n_bands)
    rate = np.array([base.y_seen[base.approved][band_acc == b].mean() for b in range(n_bands)])
    band_rej = score_bands(s_acc, s[~base.approved], n_bands)
    return float(base.book.y[~base.approved].sum() / rate[band_rej].sum())


def parcelled_scorecard(base: Base, uplift: float) -> np.ndarray:
    """Parcelling at a given uplift: infer the declined applicants' outcomes, stack, refit."""
    s = predict_pd(base.coef_acc, base.book.z)
    p_bad = parcel_bad_rates(s[base.approved], base.y_seen[base.approved], s[~base.approved],
                             BANDS, uplift)
    X, y, w = stack_parcelled(base.book.z[base.approved], base.y_seen[base.approved],
                              base.book.z[~base.approved], p_bad)
    return fit_logistic(X, y, w)


def _show_parcelling() -> None:
    names = ("approvals-only", "augmented", "parcel x1", "parcel x2", "parcel planted", "oracle")
    for share in (HIDDEN_SHARE, 0.0):
        base = fit_base(APPROVE_RATE, share, SEED)
        measured = planted_uplift(base)
        truth = max(1.0, measured)   # exercise 6 accepts no uplift below 1
        coefs = {"approvals-only": base.coef_acc, "augmented": augmented_scorecard(base)[0],
                 "parcel x1": parcelled_scorecard(base, 1.0),
                 "parcel x2": parcelled_scorecard(base, 2.0),
                 "parcel planted": parcelled_scorecard(base, truth), "oracle": base.coef_all}
        print(f"hidden share {share:.0%}: the declined really default at {measured:.2f} times the "
              f"rate of approvals in their score band")
        print(f"  {'scorecard':<16}{'Gini':>8}{'A/E all':>9}{'A/E declined':>14}")
        for name in names:
            r = judge(coefs[name], base.test, base.approved_test)
            print(f"  {name:<16}{r['gini']:>8.3f}{r['ae']:>9.2f}{r['ae_declined']:>14.2f}")
        S.setdefault("parcel", {})[share] = measured
        print()
    print("'parcel planted' uses the truth only a simulation can supply: it shows what a perfect "
          "uplift would buy, and how far an uplift that is guessed (x1, x2) lands from it.")
    print("It is handed the declined applicants' total default count, and a logistic fit with an "
          "intercept reproduces the default count of the data it is fitted on, so its A/E on "
          "everyone sits near 1 by construction. Its A/E on the declined alone and its Gini are "
          "what it actually earned.")


_try("parcelling", _show_parcelling,
     needs=("exercise 1", "exercise 2", "exercise 4", "exercise 6", "exercise 7"), heavy=True)

## 13. Exercise 8 — `banded_bounds()`

The third answer stops trying to estimate the declined applicants' default rate and states
what each assumption about it implies for the **through-the-door default rate**, the number
pricing and capital are built from. With the same bands as parcelling, suppose the declined
applicants' default rate in a band lies between `floor_ratio` and `ceil_ratio` times the
approvals' rate there (each capped at 1). The through-the-door rate is then known to lie
between two values: the approvals' observed defaults plus the declined applicants' *lowest*
allowed defaults, and the same with their *highest*, each over every applicant.

The two ends of the dial are worth knowing. `floor_ratio = 0` and `ceil_ratio = infinity` is
the classic **worst-case bound**: no assumption at all, so the declined applicants might all
have repaid or all have defaulted, and the interval is as wide as the declined share is large.
Every narrower interval is bought with an assumption, and the exercise makes you state it.

<details><summary>💡 Hint 1 — what to think about</summary>

In a band where the approvals never defaulted, what does `ceil_ratio` times their rate give,
and is that a sensible upper limit for the declined applicants there? What does `infinity`
times zero give in numpy, and what should the upper limit be when the ceiling is infinite?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Reuse the band machinery of exercise 6. For each declined applicant work out the lowest and the
highest default probability the assumption allows, add the approvals' observed defaults to each
total and divide by the number of applicants of both kinds. An infinite ceiling means a
probability of 1, whatever the band's rate. Reject `floor_ratio` below 0 or not finite,
a `ceil_ratio` below `floor_ratio`, and the same malformed inputs as exercise 6.
</details>

In [ ]:
def banded_bounds(score_acc: np.ndarray, y_acc: np.ndarray, score_rej: np.ndarray, n_bands: int,
                  floor_ratio: float, ceil_ratio: float) -> tuple[float, float]:
    """Bounds (low, high) on the default rate of ALL applicants, approved and declined.

    Bands are the approvals' score bands, as in `parcel_bad_rates`. A declined applicant's default
    probability is assumed to lie in `[min(1, floor_ratio * r), min(1, ceil_ratio * r)]`, with `r`
    the approvals' default rate in its band; an infinite `ceil_ratio` means the top of the
    interval is 1 whatever `r` is. The bounds are the approvals' defaults plus the sum of the
    declined applicants' lowest, then highest, allowed probabilities, over `n_acc + n_rej`.

    Raises ValueError for `floor_ratio` below 0 or not finite, `ceil_ratio` below `floor_ratio`
    (or NaN), and for the malformed inputs `parcel_bad_rates` rejects (except `uplift`).

    Example (same data as exercise 6: the declined sit in bands with rates 0, 0.5 and 0.5):
        >>> lo, hi = banded_bounds(np.array([1.0, 2.0, 3.0, 4.0]), np.array([0, 0, 1, 0]),
        ...                        np.array([0.5, 2.5, 9.0]), 2, 1.0, 2.0)
        >>> round(lo, 4), round(hi, 4)
        (0.2857, 0.4286)

    Returns:
        (low, high) as Python floats, with low <= high.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_banded_bounds() -> None:
    sa = np.array([1.0, 2.0, 3.0, 4.0])
    ya = np.array([0, 0, 1, 0])
    sr = np.array([0.5, 2.5, 9.0])
    lo, hi = banded_bounds(sa, ya, sr, 2, 1.0, 2.0)
    assert abs(lo - 2 / 7) < 1e-12 and abs(hi - 3 / 7) < 1e-12, (
        f"got ({lo:.4f}, {hi:.4f}), expected (0.2857, 0.4286). The approvals contributed 1 "
        "default; the declined applicants add 0 + 0.5 + 0.5 = 1 at the floor and 0 + 1 + 1 = 2 "
        "at the ceiling (2 * 0.5 capped at 1); the divisor is all 7 applicants")
    lo, hi = banded_bounds(sa, ya, sr, 2, 0.0, np.inf)
    assert abs(lo - 1 / 7) < 1e-12 and abs(hi - 4 / 7) < 1e-12, (
        f"no-assumption bounds gave ({lo:.4f}, {hi:.4f}), expected (0.1429, 0.5714): all three "
        "declined applicants repay at the floor, all three default at the ceiling")
    ya0 = np.array([0, 0, 0, 0])
    lo, hi = banded_bounds(sa, ya0, sr, 2, 1.0, np.inf)
    assert np.isfinite(hi) and abs(hi - 3 / 7) < 1e-12, (
        f"a band whose approvals never defaulted gave a high of {hi}; with an infinite ceiling "
        "the declined applicants there can all default (probability 1), and infinity times zero "
        "is NaN, not 1")
    try:
        banded_bounds(sa, ya, sr, 2, 2.0, 1.0)
    except ValueError:
        pass
    else:
        raise AssertionError("a ceiling below the floor must raise ValueError")
    print("exercise 8 looks right — the bounds are the assumption, written as an interval")

In [ ]:
_try("exercise 8", _check_banded_bounds)

In [ ]:
def _show_bounds() -> None:
    base = fit_base(APPROVE_RATE, HIDDEN_SHARE, SEED)
    s = predict_pd(base.coef_acc, base.book.z)
    truth = float(base.book.y.mean())
    n_rej = int((~base.approved).sum())
    rows = ((0.0, np.inf, "no assumption (declined: anywhere from all good to all bad)"),
            (1.0, np.inf, "declined at least as risky as approvals in their band"),
            (1.0, 2.0, "...and at most 2 times as risky"),
            (1.0, 4.0, "...and at most 4 times as risky"))
    print(f"{'assumption':<62}{'low':>8}{'high':>8}{'width':>8}  holds the truth?")
    inside = []
    for floor, ceil, label in rows:
        lo, hi = banded_bounds(s[base.approved], base.y_seen[base.approved], s[~base.approved],
                               BANDS, floor, ceil)
        ok = lo <= truth <= hi
        inside.append(ok)
        print(f"{label:<62}{lo:>8.1%}{hi:>8.1%}{hi - lo:>8.1%}  {'yes' if ok else 'NO'}")
    lo1, _ = banded_bounds(s[base.approved], base.y_seen[base.approved], s[~base.approved],
                           BANDS, 1.0, np.inf)
    forecast = float(s.mean())
    print(f"\nthrough-the-door default rate: the approvals-only scorecard forecasts "
          f"{forecast:.1%}, the approvals themselves defaulted at "
          f"{base.book.y[base.approved].mean():.1%}, and the truth is {truth:.1%}.")
    print(f"the no-assumption interval is {n_rej / N_BOOK:.1%} wide, which is the declined share "
          f"of the book: it cannot be narrower without an assumption.")
    if forecast < lo1:
        print(f"the scorecard's forecast is BELOW the lowest value that 'the declined are at least "
              f"as risky as approvals' allows ({lo1:.1%}). No label for a declined applicant was "
              "needed to see that something is wrong.")
    n_ok = sum(inside)
    print(f"{n_ok} of {len(rows)} assumptions printed an interval that holds the truth; an "
          "interval that misses it is an assumption the declined applicants contradict.")


_try("bounds", _show_bounds, needs=("exercise 1", "exercise 8"), heavy=True)

## 14. Which remedy recovers what

One book, and one scenario, is one draw. The last measurement repeats the comparison for each
hidden share of the old policy's score, at the approval rate of the scenario, over
`REPLICATES` books from fixed seeds. For every scorecard it prints the A/E on everybody, with
the range across books, the Gini, and the **share of the log A/E gap closed**: if the
approvals-only scorecard's A/E is `a` and a remedy's is `r`, that is `1 - log(r) / log(a)`.
A value of 100% means the remedy reached an A/E of 1; a negative value means it made things
worse; a value above 100% means it overshot to an A/E below 1. A book whose approvals-only A/E
is 1.05 or less has no gap worth a share and is left out of that average (the cell says if any
was). 'parcel planted' is told the answer, so its A/E near 1 is by construction; the verdict on
which remedy lands nearest 1 is taken among the three a lender could actually run.

In [ ]:
MIN_GAP = 1.05   # an approvals-only A/E at or below this leaves no gap worth a share


def _sweep_remedies() -> None:
    names = ("approvals-only", "augmented", "parcel x1", "parcel x2", "parcel planted", "oracle")
    table: dict[float, dict[str, list[tuple[float, float]]]] = {}
    planted: dict[float, list[float]] = {}
    for share in SHARES:
        table[share] = {n: [] for n in names}
        planted[share] = []
        for k in range(1, REPLICATES + 1):
            base = fit_base(APPROVE_RATE, share, _book_seed(k))
            truth = max(1.0, planted_uplift(base))
            planted[share].append(truth)
            coefs = {"approvals-only": base.coef_acc, "augmented": augmented_scorecard(base)[0],
                     "parcel x1": parcelled_scorecard(base, 1.0),
                     "parcel x2": parcelled_scorecard(base, 2.0),
                     "parcel planted": parcelled_scorecard(base, truth),
                     "oracle": base.coef_all}
            for n in names:
                r = judge(coefs[n], base.test, base.approved_test)
                table[share][n].append((r["ae"], r["gini"]))
    S["remedies"] = table
    closed: dict[float, dict[str, float]] = {}
    for share in SHARES:
        print(f"hidden share {share:.0%} of the old policy's score "
              f"(planted uplift of the declined, mean of {REPLICATES} books: "
              f"{np.mean(planted[share]):.2f})")
        print(f"  {'scorecard':<16}{'A/E on everybody':>26}{'Gini':>8}{'log gap closed':>16}")
        closed[share] = {}
        for n in names:
            ae = np.array([a for a, _ in table[share][n]])
            gini = np.mean([g for _, g in table[share][n]])
            acc = np.array([a for a, _ in table[share]["approvals-only"]])
            usable = acc > MIN_GAP
            share_closed = (float(np.mean(1.0 - np.log(ae[usable]) / np.log(acc[usable])))
                            if usable.any() else float("nan"))
            closed[share][n] = share_closed
            gap = "" if n == "approvals-only" else f"{share_closed:>16.0%}"
            print(f"  {n:<16}{f'{ae.mean():.2f} ({ae.min():.2f}-{ae.max():.2f})':>26}"
                  f"{gini:>8.3f}{gap:>16}")
        print()
    benign, worst = SHARES[0], SHARES[1:]
    print(f"augmentation closes {closed[benign]['augmented']:.0%} of the calibration gap when the "
          f"policy used only the features, and {np.mean([closed[s]['augmented'] for s in worst]):.0%} "
          f"on average over the columns where it used the hidden variable.")
    print(f"parcelling at an uplift of 1 closes {np.mean([closed[s]['parcel x1'] for s in SHARES]):.0%} "
          f"on average; at the planted uplift, which a lender would not know, "
          f"{np.mean([closed[s]['parcel planted'] for s in SHARES]):.0%}.")
    runnable = ("augmented", "parcel x1", "parcel x2")
    wins = [min(runnable, key=lambda n: abs(np.log(np.mean([a for a, _ in table[s][n]]))))
            for s in SHARES]
    print("among the remedies a lender could run (augmentation, parcelling at x1 or x2), the one "
          "whose mean A/E lands nearest 1, column by column: "
          + ", ".join(f"{s:.0%} -> {w}" for s, w in zip(SHARES, wins)) + ".")
    left_out = sum(int(np.sum(np.array([a for a, _ in table[s]["approvals-only"]]) <= MIN_GAP))
                   for s in SHARES)
    if left_out:
        print(f"{left_out} book(s) whose approvals-only A/E was {MIN_GAP} or less were left out of "
              "the 'log gap closed' averages: there was no gap worth a share.")


_try("which remedy recovers what", _sweep_remedies,
     needs=("exercise 1", "exercise 2", "exercise 4", "exercise 6", "exercise 7"), limit=100,
     heavy=True)

Read the table against what each method is able to see. Augmentation sees the features, so by
construction it can correct the extrapolation they cause and should help most in the columns
where the old policy was a function of them. Parcelling asks for an uplift, a statement about
the *declined* applicants, the part of the population the data are silent on: it can reach
their true rate if the uplift is right and cannot if it is not. Bounds report the uncertainty
and decline to resolve it. The printed lines above say which of these your run supports. A
validator who asks which remedy you used is asking which assumption you made, and a model file
that cannot name one has made one by accident.

## 15. Common mistakes

- **Reporting the lender's own test.** A scorecard's A/E on a holdout of *approvals* describes
  the population it was fitted on. Its Gini on that holdout is narrowed by the same selection.
  Neither number is the through-the-door performance.
- **Calling selection "a sampling problem".** More approvals from the same policy give a more
  precise answer to the wrong question. Only information about the declined, a stated
  assumption, or outcome data on some applicants the policy would have declined can move it.
- **Thresholding with `<=` on a coarse score.** It approves every applicant tied at the line, so
  the policy you simulated is not the one you meant, and the approved fraction is not what you
  report.
- **Truncating `rate * n`.** Floating point makes some products fall a hair under the integer
  they should be. Round to the nearest applicant.
- **Capping nothing.** An approval the old policy almost never let in can carry a weight of a
  thousand and decide the fit. Report the effective sample size beside every weighted fit.
- **Cutting the declined applicants into their own bands.** The bands must come from the
  approvals, or the declined applicants' rate is compared with itself.
- **Treating an uplift of 1 as neutral.** It is the assumption that the declined behave
  exactly like the approvals in the same band, and it is the most optimistic one on offer.
- **Letting an outcome reach a remedy.** Everything a remedy reads must be the lender's
  data: features for everyone, outcomes for the approvals only. The notebook zeroes the declined
  applicants' outcomes before any remedy sees them for exactly this reason.

Run the cell below to watch the first two of those happen, with numbers computed on the spot.

In [ ]:
def _show_two_mistakes() -> None:
    sizes, percents = (100, 1000), range(1, 100)
    wrong = [(n, k) for n in sizes for k in percents if int(k / 100 * n) != n * k // 100]
    print(f"truncating rate * n: of the {len(percents) * len(sizes)} whole-percent rates on "
          + " and ".join(f"{n:,}" for n in sizes) + f" applicants, {len(wrong)} approve one "
          "applicant too few, for example "
          + ", ".join(f"n={n}, rate={k / 100:.2f}" for n, k in wrong[:3]) + ".")
    scores = np.array([1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 3])
    k = 6
    cutoff = np.sort(scores)[k - 1]
    print(f"a `score <= cutoff` test on {scores.size} scores with {np.unique(scores).size} distinct "
          f"values, asked for {k} approvals, approves {int((scores <= cutoff).sum())}.")


_show_two_mistakes()

## 16. Self-check

1. A scorecard trained on a lender's approvals reports an A/E close to 1.00 on a holdout drawn
   from those approvals. The right reading is:
   - (a) it will be calibrated on the full through-the-door population too
   - (b) it is calibrated for the population it was fitted on, which says nothing about the
         applicants who were declined
   - (c) the old policy was unbiased, since the holdout agrees with the training data

2. A lender's old policy was a hard threshold on one recorded score, with no overrides.
   You fit a model of approval from the features and weight the approvals by the inverse
   of its fitted probability. What do you expect?
   - (a) the weights correct the selection completely, because the approval model is exact
   - (b) the weights correct it once the cap is raised high enough
   - (c) almost every approval gets a weight near 1, because nobody on the declined side of the
         line was ever approved, and the reweighting changes little

3. A colleague says the hidden-variable problem will fade once the book of approved loans is
   large enough. The best reply is:
   - (a) selection on a variable the data do not contain is a bias, so a larger book of
         approvals estimates the wrong quantity more precisely
   - (b) a larger book approves riskier applicants, which dilutes the bias
   - (c) that is true if the noise in the book is independent

4. You choose an uplift of 1 for parcelling "because it is the choice that adds no assumption".
   - (a) correct: it uses only the data, so nothing has been assumed
   - (b) it is an assumption, that the declined behave exactly like the approvals in the same
         band, and nothing in the lender's data can test it
   - (c) it is the conservative choice, because it overstates the declined applicants' risk

5. The worst-case bounds on the through-the-door default rate are wide. The honest ways to
   narrow them are:
   - (a) a larger book of approved loans, or a better scorecard on the approvals
   - (b) rerunning the sweep with more replicate books
   - (c) a stated assumption about the declined applicants, or outcome data on some applicants
         the old policy would have declined

Answers, with the reasoning, come with this lesson's worked solution when you enrol on Synapsa.

One last cell, and it is the deliverable. A scorecard hands on with the sentence about its
selection attached, or someone will later quote its A/E as though it described the people it
never saw.

In [ ]:
def _selection_note() -> None:
    damage = S["damage"]
    a, o = damage["approved"], damage["oracle"]
    truth = S["parcel"][HIDDEN_SHARE]
    print("SELECTION NOTE — hand this on with the scorecard")
    print(f"  fitted on    approvals only: {APPROVE_RATE:.0%} of a synthetic through-the-door book "
          f"of {N_BOOK:,} applicants")
    print(f"  measured     on a fresh sample of {N_TEST:,} applicants with every outcome known")
    print(f"  as measured  Gini {a['gini']:.3f} on everyone against {o['gini']:.3f} for a "
          f"scorecard trained on everyone; A/E {a['ae']:.2f} on everyone, {a['ae_declined']:.2f} "
          "on the declined applicants alone")
    print(f"  its own test A/E {a['ae_approved']:.2f} on approvals: the number that looks fine and "
          "describes the wrong population")
    print(f"  remedies     augmentation, parcelling and bounds, measured above; the declined "
          f"applicants really defaulted at {truth:.2f} times the rate of approvals in their band")
    print("  known gaps   one scenario and one generator; the hidden variable is planted and "
          "no remedy here can see it; the uplift of a real lender is a judgement, not a measurement")


_try("selection note", _selection_note,
     needs=("exercise 1", "exercise 2", "exercise 4", "exercise 6", "exercise 7"), heavy=True)

## What you built, and where it goes next

You now have a way to ask the question a scorecard's own holdout cannot answer: *how does this
model do on the people it never saw?* A policy simulator that knows the counterfactual, a
tie-aware Gini, a calibration table, augmentation weights with their price in effective sample
size, parcelling with an uplift you must name, and bounds that print the assumption as an
interval. The remaining modules of this programme inherit the selection you just measured:
the choice between a scorecard and boosting is made on approvals, the reasons you owe a
declined applicant come from a model fitted on the people who were not declined, and the
monitoring of a live portfolio watches only the loans that were made.

The habit to carry forward: **before reporting what a model learned, say who was allowed to
teach it.** The honest figure is the one that names the population it was measured on. As of
2 October 2026 the attachment to the US model risk guidance SR 26-2 lists, among the forms
model testing may take, "a critical assessment of data quality, relevance, and inputs": the
question of who taught the model is a question about the relevance of its data, and this lesson
is a way of putting a number on it.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_impose_policy),
                              ("exercise 2", _check_auc_gini),
                              ("exercise 3", _check_calibration_bands),
                              ("exercise 4", _check_augmentation_weights),
                              ("exercise 5", _check_effective_sample_size),
                              ("exercise 6", _check_parcel_bad_rates),
                              ("exercise 7", _check_stack_parcelled),
                              ("exercise 8", _check_banded_bounds)):
            _try(_name, _check)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.